In [ ]:
import trenchripper as tr
import numpy as np
import pandas as pd
import dask.dataframe as dd

from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="3:00:00",
    local=False,
    n_workers=20,
    n_workers_min=20,#200,
    memory="8GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

# dask_controller = tr.trcluster.dask_controller(
#     walltime="2:00:00",
#     local=False,
#     n_workers=50,
#     n_workers_min=50,
#     memory="8GB",
#     working_directory=dask_wd,
# )
# dask_controller.startdask()

In [ ]:
dask_controller.daskclient

In [150]:
# df_obs_growth = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Growth_Observations",calculate_divisions=True)
# df_obs_obs = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Observations",calculate_divisions=True)
# df_obs_cell_cycle = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Cell_Cycle",calculate_divisions=True)
df_bar = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Final_Barcodes_df", engine="pyarrow",calculate_divisions=True)

# Get also lineage information
# df_lineage = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/lineage", engine="pyarrow",calculate_divisions=True)
# df = pd.read_hdf("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/barcode_output_df.hdf5")
bc_kymo_metadata = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/kymograph/metadata", calculate_divisions=True)


### Understanding how the tabular data is structured

- `df_obs_growth`, `df_obs_obs`, `df_obs_cell_cycle`, and `df_bar` all are essentially indexed by `phenotype trenchid`.

- `df_bar` has all the barcode information, and the mapping to the phenotype kymographs. In this table:
    - `phenotype trenchid` is the trench index from the phenotyping data. This can be used to map to the lineage dataframe.
    - `trenchid` is the trench index from the FISH data. It can be used to map to the FISH kymograph metadata to obtain the file indices and file trench indices.
    - `File Index`, `File Trench Index` are the indices for the **phenotype** kymographs.
    - `Barcode` is the barcode sequence from the phenotype, which is found from the FISH images. It is arranged by channel, and then by cycle number. That is: $(\text{ch}_1, \text{cyc}_1), (\text{ch}_1, \text{cyc}_2), ..., (\text{ch}_1, \text{cyc}_{15}), (\text{ch}_2, \text{cyc}_1),..., (\text{ch}_2, \text{cyc}_{15})$.
    - `barcode` is the barcode sequence inferred from Nanopore sequencing. It is arranged by bit number in the actual RNA barcode. There is a lookup table that maps positions in barcode RNA to cycle order. In my case it is:

    ```
    cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,
                  2,3,8,9,14,15,20,21,26,27,17,22,23,28,29],
    ```


In [151]:
# Merging to get lookup df 
bar_single_trench= (df_bar
                    .loc[:, ['phenotype trenchid', 'File Index', 'File Trench Index', 'trenchid', 'Barcode', 'barcode', 'barcodeid']]
                    .groupby('phenotype trenchid')
                    .first()
                    .compute()
)
bc_kymo_single_trench = (bc_kymo_metadata
                         .loc[:,['File Index', 'File Trench Index', 'trenchid']]
                         .groupby('trenchid')
                         .first()
                         .compute()
)

df_lookup = (bc_kymo_single_trench
             .reset_index()
             .merge(bar_single_trench.reset_index(), on='trenchid', how='inner', suffixes=('_bc', '_pheno'))
)

In [165]:
### QC segmentation
import h5py
import matplotlib.pyplot as plt
import numpy as np

def show_kymograph(
        path_prefix,
        path_kymo_dir_prefix,
        channels,
        cmap,
        kymo_file_index,
        trench_index,
        t_indices,
        figsize):
    with h5py.File(path_prefix + path_kymo_dir_prefix + str(kymo_file_index) + ".hdf5", "r") as infile:
        for ch in channels:
            img = infile[ch][trench_index][t_indices[0]]
            for ind in t_indices[1:]:
                img_ind = infile[ch][trench_index][ind]
                img = np.concatenate((img, img_ind), axis=1)
            plt.figure(figsize=figsize)
            # Do not show axes ticks
            plt.xticks([])
            plt.yticks([])
            plt.imshow(img, cmap=cmap, aspect=1)
            # Annotate the kymo_file_index and the trench index at the bottom left corner
            plt.annotate(f'kymo_file_index: {kymo_file_index}, trench index: {trench_index}', (10, 15), color='white')
            plt.show()
            plt.close()

In [ ]:
row = 274
pheno_kymo_file_index, pheno_trench_index, bc_kymo_file_index, bc_trench_index = df_lookup.loc[row, ['File Index_pheno', 'File Trench Index_pheno', 'File Index_bc', 'File Trench Index_bc']]
path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/"
path_kymo_dir_prefix = "kymograph/kymograph_"
channels = ["mCherry"]
cmap = 'gray'
t_indices = np.arange(0, 61)
# pheno_kymo_file_index = 0
# pheno_trench_index = 42
figsize = (15, 30)
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, pheno_kymo_file_index, pheno_trench_index, t_indices, figsize)
# path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/"
# path_kymo_dir_prefix = "fluorsegmentation/segmentation_"
# channels = ["data"]
# cmap = "jet"
# show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, pheno_kymo_file_index, pheno_trench_index, t_indices, figsize)
# bc_kymo_file_index = 0
# bc_trench_index = 73
figsize = (3, 6)
path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/"
path_kymo_dir_prefix = "kymograph/kymograph_"
channels = ["BF", "RFP", "Cy5"]
cmap = "gray"
t_indices = np.arange(0,15)
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, bc_kymo_file_index, bc_trench_index, t_indices, figsize)

In [ ]:
df_lookup

In [166]:
### QC segmentation
import h5py
import matplotlib.pyplot as plt
import numpy as np

def show_kymograph(
        path_prefix,
        path_kymo_dir_prefix,
        channels,
        cmap,
        kymo_file_index,
        trench_index,
        t_indices,
        figsize,
        Barcode = None,
        barcode_phe = None #translated from seq barcode
        ):
    
    n_cycles_per_ch = 15 # TODO GENERALIZE
    with h5py.File(path_prefix + path_kymo_dir_prefix + str(kymo_file_index) + ".hdf5", "r") as infile:
        for i_ch, ch in enumerate(channels):
            img = infile[ch][trench_index][t_indices[0]]
            for ind in t_indices[1:]:
                img_ind = infile[ch][trench_index][ind]
                img = np.concatenate((img, img_ind), axis=1)
            plt.figure(figsize=figsize)
            # Do not show axes ticks
            plt.xticks([])
            plt.yticks([])
            plt.imshow(img, cmap=cmap, aspect=1)
            # Annotate the kymo_file_index and the trench index at the bottom left corner
            plt.annotate(f'kymo_file_index: {kymo_file_index}, trench index: {trench_index}', (10, 15), color='white')
            # Annotate the cycle number at the bottom of each slice
            for i, ind in enumerate(t_indices):
                plt.annotate(f'{ind}', (i*13+13/4, 160), color='white', fontsize=8) # 13 is the width of a slice
            
            # Annotate the Barcode bit at the bottom of each slice
            if Barcode is not None:
                
                for i, bit in enumerate(Barcode[i_ch*n_cycles_per_ch:(i_ch+1)*n_cycles_per_ch]):
                    plt.annotate(f'{bit}', (i*13+13/4, 140), color='yellow', fontsize=8) # 13 is the width of a slice

            # Annotate the barcode bit at the bottom of each slice
            if barcode_phe is not None:
                for i, bit in enumerate(barcode_phe[i_ch*n_cycles_per_ch:(i_ch+1)*n_cycles_per_ch]):
                    plt.annotate(f'{bit}', (i*13+13/4, 120), color='red', fontsize=8) # 13 is the width of a slice

            plt.show()
            plt.close()

In [167]:
def map_bc_to_pheno(barcode,
                    cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,
                                  2,3,8,9,14,15,20,21,26,27,17,22,23,28,29],
                    ):
    barcode_phe = [-1]*30
    for i, bit in enumerate(barcode):
        # print(i, bit)
        # Find the position of i in cycleorder
        i_phe = cycleorder.index(i)
        barcode_phe[i_phe] = bit
    barcode_phe = ''.join(barcode_phe)
    return barcode_phe

In [ ]:
row = 600
pheno_kymo_file_index, pheno_trench_index, bc_kymo_file_index, bc_trench_index = df_lookup.loc[row, ['File Index_pheno', 'File Trench Index_pheno', 'File Index_bc', 'File Trench Index_bc']]
# barcode (lowercase) comes from sequencing
# Barcode (uppercase) comes from trench signal
Barcode, barcode = df_lookup.loc[row, ['Barcode', 'barcode']]
barcode_phe = map_bc_to_pheno(barcode)
figsize = (3, 6)
path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/"
path_kymo_dir_prefix = "kymograph/kymograph_"
# channels = ["BF", "RFP", "Cy5"]
channels = ["RFP", "Cy5"]
cmap = "gray"
t_indices = np.arange(0,15)
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, bc_kymo_file_index, bc_trench_index, t_indices, figsize, Barcode, barcode_phe)

In [ ]:
# TODO DO IT FROM FISH MAPPING RIGHT AFTER IT IS DONE (WITHOUT PHENOTYPE)
   # Check the quantification one for discarded ones, view them, and see what I can do better
   # Apply what I learned from classifiers
# TODO Check how Daniel makes the comparison between barcodes (e.g. to find hamming distance)

# Have a nice viewer for the phenotype and the barcodes combined (multipanel with custom arrangement)
# Filter by e.g. gRNA (I can do that manually very easily).

# Backup data!!! Reset notebooks and commit

In [ ]:
dask_controller.shutdown()

In [ ]:
path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/"
path_kymo_dir_prefix = "kymograph/kymograph_"
channels = ["mCherry"]
cmap = 'gray'
t_indices = np.arange(0, 61)
kymo_file_index = 201
trench_index = 27
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, kymo_file_index, trench_index, t_indices)
path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/"
path_kymo_dir_prefix = "fluorsegmentation/segmentation_"
channels = ["data"]
cmap = "jet"
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, kymo_file_index, trench_index, t_indices)
path_prefix = "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/"
path_kymo_dir_prefix = "fluorsegmentation/segmentation_"
channels = ["data"]
cmap = "jet"
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, kymo_file_index, trench_index, t_indices)
